# Learning MongoDB queries and index choices

## 1. Setup and Imports

In [9]:
import json
import random
from pathlib import Path

import torch

from origami import DataConfig, ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig

# For reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.9.1


## 2. Load and Explore the Data

The car dataset contains nested JSON objects with car attributes and acceptability ratings.

In [ ]:
# Load data from JSONL file
train_path = Path("../datasets/mongodb_workload_train.jsonl")
with open(train_path) as f:
    train_data = [json.loads(line) for line in f][:4000]

test_path = Path("../datasets/mongodb_workload_test.jsonl")
with open(test_path) as f:
    test_data = [json.loads(line) for line in f]

print(f"Loaded {len(train_data)} train records")
print(f"Loaded {len(test_data)} test records")

print(f"\nSample record:")
print(json.dumps(train_data[0], indent=2))

Loaded 4000 train records
Loaded 2000 test records

Sample record:
{
  "filter": {
    "SCHEDULED_ARRIVAL": {
      "$lte": "2118"
    },
    "ORIGIN_AIRPORT": {
      "$gte": "TVC"
    },
    "SCHEDULED_DEPARTURE": {
      "$lte": "0534"
    }
  },
  "sort": {
    "SCHEDULED_DEPARTURE": 1
  },
  "limit": 819,
  "projection": {
    "SCHEDULED_ARRIVAL": 1,
    "CANCELLED": 1,
    "_id": 0
  },
  "index_id": 3
}


In [16]:
# load CSV file
import pandas as pd

train_df = pd.read_csv("/Users/thomas/code/experiments/origami-data-gen/baseline-code/TabDiff/data/default/train.csv")
display(train_df.head())
train_data = train_df.to_dict(orient="records")

test_df = pd.read_csv("/Users/thomas/code/experiments/origami-data-gen/baseline-code/TabDiff/data/default/test.csv")
test_data = test_df.to_dict(orient="records")

print(f"Loaded {len(train_data)} train records")
print(f"Loaded {len(test_data)} test records")


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000.0,1,1,1,34.0,-2,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,80000.0,1,2,2,34.0,0,0,0,0,0,...,43182.0,44332.0,45440.0,2600.0,4300.0,2000.0,2000.0,2000.0,2000.0,0
2,200000.0,2,3,1,49.0,1,-2,-1,-1,-1,...,7588.0,7606.0,14053.0,0.0,2317.0,7588.0,7614.0,14053.0,0.0,0
3,20000.0,2,2,1,41.0,-1,-1,-1,-1,-1,...,0.0,7014.0,7696.0,1087.0,1140.0,0.0,7014.0,800.0,0.0,0
4,70000.0,2,1,1,36.0,2,0,0,0,0,...,65287.0,35345.0,9360.0,5000.0,3000.0,2000.0,3000.0,5000.0,0.0,0


Loaded 27000 train records
Loaded 3000 test records


## 4. Training with Custom Configuration

`OrigamiConfig` with nested `ModelConfig`, `TrainingConfig`, and `DataConfig` lets you customize model architecture, training parameters, and preprocessing options.

In [17]:
from origami.training import accuracy

# Create a custom configuration with nested structure
custom_config = OrigamiConfig(
    model=ModelConfig(
        d_model=128,  # Embedding dimension
        n_heads=4,  # Attention heads (must divide d_model)
        n_layers=4,  # Transformer layers
        d_ff=512,  # Feed-forward dimension
        dropout=0.0,  # Dropout rate
        use_continuous_head=True,
        continuous_loss_weight=-1.0,
    ),
    training=TrainingConfig(
        batch_size=100,
        learning_rate=1e-3,
        num_epochs=50,
        warmup_steps=1000,
        shuffle_keys=True,  # Data augmentation via key order shuffling
        eval_strategy="steps",
        eval_steps=100,
        eval_metrics={"acc": accuracy},
        target_key="default payment next month",
        eval_sample_size=100,
        eval_on_train=True,  # Evaluate on training data as well
        constrain_grammar=True,
        constrain_schema=True,
    ),
    data=DataConfig(
        numeric_mode="scale",  # Car dataset has no high-cardinality numerics
        max_vocab_size=2000,
        infer_schema=True,
    ),
)

print("Custom configuration:")
print(f"  d_model: {custom_config.model.d_model}")
print(f"  n_layers: {custom_config.model.n_layers}")
print(f"  batch_size: {custom_config.training.batch_size}")
print(f"  shuffle_keys: {custom_config.training.shuffle_keys}")

Custom configuration:
  d_model: 128
  n_layers: 4
  batch_size: 100
  shuffle_keys: True


In [20]:
from origami.training import TableLogCallback

# Create and train pipeline with custom config
pipeline = OrigamiPipeline(custom_config)
pipeline.fit(train_data, eval_data=test_data, callbacks=[TableLogCallback(print_every=10)], verbose=True)

print(f"\nTraining complete!")
print(f"Model parameters: {pipeline._model.get_num_parameters():,}")

KeyboardInterrupt: 

In [15]:
pipeline.evaluate(test_data, metrics={"acc": accuracy})

{'loss': 1.5894116503851754, 'acc': 0.868}

In [19]:
docs = pipeline.generate(100)

docs

[{'default payment next month': 0,
  'PAY_2': -2,
  'AGE': 44.0,
  'BILL_AMT3': -1719,
  'BILL_AMT1': 964511,
  'BILL_AMT5': 50135,
  'LIMIT_BAL': 210000.0,
  'MARRIAGE': 2,
  'PAY_4': -2,
  'PAY_AMT6': 900,
  'PAY_AMT5': 2805,
  'PAY_AMT3': 1107,
  'PAY_3': -2,
  'SEX': 1,
  'BILL_AMT6': 9199,
  'BILL_AMT4': 891586,
  'PAY_AMT4': 2432,
  'PAY_AMT2': 2013,
  'PAY_0': 1,
  'BILL_AMT2': 1609,
  'PAY_6': -2,
  'EDUCATION': 1,
  'PAY_AMT1': 1782,
  'PAY_5': -2},
 {'PAY_AMT2': 3627,
  'BILL_AMT3': 208856,
  'BILL_AMT5': 40913,
  'PAY_AMT6': 1181,
  'BILL_AMT1': 46034,
  'BILL_AMT4': 65030,
  'BILL_AMT6': 81728,
  'PAY_AMT5': 802,
  'AGE': 26.0,
  'PAY_5': 0,
  'default payment next month': 0,
  'PAY_AMT3': 2686,
  'PAY_4': 0,
  'PAY_AMT1': 10795,
  'PAY_6': 0,
  'PAY_AMT4': 1887,
  'SEX': 1,
  'PAY_2': 0,
  'MARRIAGE': 1,
  'LIMIT_BAL': 130000.0,
  'BILL_AMT2': 14840,
  'EDUCATION': 2,
  'PAY_0': 0,
  'PAY_3': 0},
 {'AGE': 40.0,
  'BILL_AMT1': 11325,
  'PAY_AMT3': 1597,
  'PAY_3': -2,
  'PA